# **Data Preprocessing**

This notebook is dedicated to cleaning the raw Tanzania Tourism dataset and preparing it for the subsequent feature engineering and machine learning stages.

### Main Objectives of the Preprocessing :


## 1. Import Required Libraries

The following libraries are used throughout the preprocessing stage:

- `Path` for handling file and directory paths.
- `sys` for managing Python's module search path.
- `NumPy` for numerical operations.
- `Pandas` for data manipulation and cleaning.

In [2]:
from pathlib import Path
import sys 

notebook_dir = Path.cwd()
project_root = notebook_dir.parent 

if str(project_root) not in sys.path:
    sys.path.insert(0, str(project_root))

from src.config import TRAIN_DATA_PATH, TEST_DATA_PATH, VARIABLE_DEFINITIONS_PATH, NUMERIC_COLS, ID_COL, TARGET_COL, TARGET_CLASSES, CATEGORICAL_COLS
from src.utils import outliers, massing_analysis
import numpy as np
import pandas as pd



## 2. Load Raw Datasets

load train dataset, test dataset, and variable definitions 

In [3]:
train_df = pd.read_csv(TRAIN_DATA_PATH)
test_df = pd.read_csv(TEST_DATA_PATH)
vb_df = pd.read_csv(VARIABLE_DEFINITIONS_PATH)


In [4]:
train_df.head(5)

,Tour_ID,country,age_group,travel_with,total_female,total_male,purpose,main_activity,info_source,tour_arrangement,...,package_accomodation,package_food,package_transport_tz,package_sightseeing,package_guided_tour,package_insurance,night_mainland,night_zanzibar,first_trip_tz,cost_category
0,tour_id1hffseyw,ITALY,45-64,With Children,0.0,2.0,Visiting Friends and Relatives,Beach Tourism,"Friends, relatives",Package Tour,...,Yes,Yes,Yes,No,No,No,0,7,Yes,High Cost
1,tour_idnacd7zag,UNITED KINGDOM,25-44,With Spouse,1.0,1.0,Leisure and Holidays,Wildlife Tourism,"Travel agent, tour operator",Package Tour,...,Yes,Yes,Yes,No,No,No,0,7,Yes,High Cost
2,tour_id62vz7e71,UNITED STATES OF AMERICA,65+,With Spouse,1.0,1.0,Leisure and Holidays,Widlife Tourism,"Travel agent, tour operator",Package Tour,...,Yes,Yes,Yes,Yes,Yes,No,6,6,Yes,Higher Cost
3,tour_idrc76tzix,RWANDA,25-44,With Spouse and Children,3.0,1.0,Leisure and Holidays,Beach Tourism,"Radio, TV, Web",Independent,...,No,No,No,No,No,No,3,0,No,Lower Cost
4,tour_idn723m0n9,UNITED STATES OF AMERICA,45-64,Alone,0.0,1.0,Leisure and Holidays,Widlife Tourism,"Travel agent, tour operator",Package Tour,...,Yes,Yes,Yes,No,Yes,Yes,7,0,Yes,Higher Cost


In [5]:
train_df.shape


(18506, 21)

In [6]:
test_df.head(5)

,Tour_ID,country,age_group,travel_with,total_female,total_male,purpose,main_activity,info_source,tour_arrangement,package_transport_int,package_accomodation,package_food,package_transport_tz,package_sightseeing,package_guided_tour,package_insurance,night_mainland,night_zanzibar,first_trip_tz
0,tour_idynufedne,KOREA,25-44,Alone,0.0,1.0,Leisure and Holidays,Widlife Tourism,Others,Independent,No,No,No,No,No,No,No,7,4,Yes
1,tour_id9r3y5moe,UNITED KINGDOM,45-64,With Children,1.0,1.0,Leisure and Holidays,Conference Tourism,"Travel agent, tour operator",Package Tour,Yes,Yes,Yes,Yes,Yes,Yes,Yes,7,0,Yes
2,tour_idf6itml6g,ITALY,25-44,With Spouse,1.0,1.0,Leisure and Holidays,Beach Tourism,"Travel agent, tour operator",Package Tour,Yes,Yes,No,Yes,No,No,No,0,6,Yes
3,tour_id99u4znru,KENYA,25-44,Alone,0.0,1.0,Other,Beach Tourism,"Radio, TV, Web",Independent,No,No,No,No,No,No,No,3,4,No
4,tour_idj4i9urbx,ZAMBIA,25-44,Alone,0.0,1.0,Business,Widlife Tourism,"Radio, TV, Web",Independent,No,No,No,No,No,No,No,6,0,No


In [7]:
test_df.shape


(6169, 20)

In [8]:
vb_df

,Column Name,Definition
0,id,Unique identifier for each tourist
1,country,The country a tourist coming from.
2,age_group,The age group of a tourist.
3,travel_with,The relation of people a tourist travel with t...
4,total_female,Total number of females
5,total_male,Total number of males
6,purpose,The purpose of visiting Tanzania
7,main_activity,The main activity of tourism in Tanzania
8,infor_source,The source of information about tourism in Tan...
9,tour_arrangment,The arrangment of visiting Tanzania


## 3. Inspect Data Quality

Review data types, missing values, duplicate records, and outliers before investigating and addressing each issue separately.

**Main points :** 
- Missing Values
- Duplicate Records
- Data Types
- Data Consistency
- Invalid Values

In [9]:
print('train dataframe')
display(train_df.info())
print('_'*50,'\n') 
print('test dataframe')
display(test_df.info())

train dataframe
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 18506 entries, 0 to 18505
Data columns (total 21 columns):
 #   Column                 Non-Null Count  Dtype  
---  ------                 --------------  -----  
 0   Tour_ID                18506 non-null  object 
 1   country                18506 non-null  object 
 2   age_group              18506 non-null  object 
 3   travel_with            17431 non-null  object 
 4   total_female           18504 non-null  float64
 5   total_male             18500 non-null  float64
 6   purpose                18506 non-null  object 
 7   main_activity          18506 non-null  object 
 8   info_source            18506 non-null  object 
 9   tour_arrangement       18506 non-null  object 
 10  package_transport_int  18506 non-null  object 
 11  package_accomodation   18506 non-null  object 
 12  package_food           18506 non-null  object 
 13  package_transport_tz   18506 non-null  object 
 14  package_sightseeing    18506 non-null 

None

__________________________________________________ 

test dataframe
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 6169 entries, 0 to 6168
Data columns (total 20 columns):
 #   Column                 Non-Null Count  Dtype  
---  ------                 --------------  -----  
 0   Tour_ID                6169 non-null   object 
 1   country                6169 non-null   object 
 2   age_group              6169 non-null   object 
 3   travel_with            5808 non-null   object 
 4   total_female           6167 non-null   float64
 5   total_male             6168 non-null   float64
 6   purpose                6169 non-null   object 
 7   main_activity          6169 non-null   object 
 8   info_source            6169 non-null   object 
 9   tour_arrangement       6169 non-null   object 
 10  package_transport_int  6169 non-null   object 
 11  package_accomodation   6169 non-null   object 
 12  package_food           6169 non-null   object 
 13  package_transport_tz   6169 non-null   o

None

In [10]:
print("Missing Values in train dataset : ")
display(massing_analysis(train_df))
print('_'*50,'\n') 
print("Missing Values in test dataset : ")
display(massing_analysis(test_df))


Missing Values in train dataset : 


,number of missing values,percentage of missing values
Tour_ID,,
country,,
age_group,,
travel_with,1075.0,5.81
total_female,2.0,0.01
total_male,6.0,0.03
purpose,,
main_activity,,
info_source,,
tour_arrangement,,


__________________________________________________ 

Missing Values in test dataset : 


,number of missing values,percentage of missing values
Tour_ID,,
country,,
age_group,,
travel_with,361.0,5.85
total_female,2.0,0.03
total_male,1.0,0.02
purpose,,
main_activity,,
info_source,,
tour_arrangement,,


In [11]:
print("Duplicate Records in train dataset : ",train_df.duplicated().sum())
print("Duplicate Records in test dataset : ",test_df.duplicated().sum())


Duplicate Records in train dataset :  0
Duplicate Records in test dataset :  0


In [12]:
print("outliers for train dataset ")
outliers(train_df)
print('_'*50,'\n') 
print("outliers for test dataset ")
outliers(test_df)


outliers for train dataset 
total_female | outliers : 946 - 5.11%
total_male | outliers : 6692 - 36.16%
night_mainland | outliers : 1148 - 6.20%
night_zanzibar | outliers : 729 - 3.94%
__________________________________________________ 

outliers for test dataset 
total_female | outliers : 320 - 5.19%
total_male | outliers : 2204 - 35.73%
night_mainland | outliers : 383 - 6.21%
night_zanzibar | outliers : 271 - 4.39%


In [13]:
train_df[NUMERIC_COLS].describe()

,total_female,total_male,night_mainland,night_zanzibar
count,18504.000000,18500.000000,18506.000000,18506.000000
mean,0.936230,0.998757,9.141576,2.493516
std,1.215582,1.173177,14.127449,5.275156
min,0.000000,0.000000,0.000000,0.000000
25%,0.000000,1.000000,3.000000,0.000000
50%,1.000000,1.000000,6.000000,0.000000
75%,1.000000,1.000000,11.000000,4.000000
max,49.000000,58.000000,365.000000,240.000000


- no negative values in total_female, total_male, night_mainland and night_zanzibar.

## 4. Handle Missing Values
- select missing values in dataset and understand its cause

In [14]:
# fast missing values analysis 

def fmva(train_df, test_df=None):
    print("Missing Values in train dataset : ")
    display(massing_analysis(train_df))
    if test_df is not None:
        print('_'*50,'\n') 
        print("Missing Values in test dataset : ")
        display(massing_analysis(test_df))


In [15]:
fmva(train_df, test_df)

Missing Values in train dataset : 


,number of missing values,percentage of missing values
Tour_ID,,
country,,
age_group,,
travel_with,1075.0,5.81
total_female,2.0,0.01
total_male,6.0,0.03
purpose,,
main_activity,,
info_source,,
tour_arrangement,,


__________________________________________________ 

Missing Values in test dataset : 


,number of missing values,percentage of missing values
Tour_ID,,
country,,
age_group,,
travel_with,361.0,5.85
total_female,2.0,0.03
total_male,1.0,0.02
purpose,,
main_activity,,
info_source,,
tour_arrangement,,


In [16]:
train_df['travel_with'].unique()

array(['With Children', 'With Spouse', 'With Spouse and Children',
       'Alone', nan, 'With Other Friends/Relatives'], dtype=object)

In [17]:
def fill_Alone_values(df):
    mask_alone = (df['total_female'] + df['total_male']) <= 1 
    mask_nan = df['travel_with'].isna()
    mask = mask_nan & mask_alone
    df.loc[mask, 'travel_with'] = 'Alone'

fill_Alone_values(train_df)
fill_Alone_values(test_df)

In [18]:
fmva(train_df, test_df)

Missing Values in train dataset : 


,number of missing values,percentage of missing values
Tour_ID,,
country,,
age_group,,
travel_with,28.0,0.15
total_female,2.0,0.01
total_male,6.0,0.03
purpose,,
main_activity,,
info_source,,
tour_arrangement,,


__________________________________________________ 

Missing Values in test dataset : 


,number of missing values,percentage of missing values
Tour_ID,,
country,,
age_group,,
travel_with,7.0,0.11
total_female,2.0,0.03
total_male,1.0,0.02
purpose,,
main_activity,,
info_source,,
tour_arrangement,,


In [19]:
# one male and female
def omf(df):
    mask_nan = df['travel_with'].isna()
    mask_female = df['total_female'] ==1 
    mask_male   = df['total_male'] == 1 

    omale_and_ofemale_train = df[mask_nan & mask_female & mask_male]
    return omale_and_ofemale_train

print('train dataframe :')
display(omf(train_df))
print('test dataframe :')
display(omf(test_df))


train dataframe :


,Tour_ID,country,age_group,travel_with,total_female,total_male,purpose,main_activity,info_source,tour_arrangement,...,package_accomodation,package_food,package_transport_tz,package_sightseeing,package_guided_tour,package_insurance,night_mainland,night_zanzibar,first_trip_tz,cost_category
5926,tour_idrc9ck93b,ITALY,25-44,NaN,1.0,1.0,Leisure and Holidays,Wildlife Tourism,"Travel agent, tour operator",Package Tour,...,Yes,Yes,Yes,Yes,Yes,No,9,0,Yes,Higher Cost
8745,tour_idqgl7rdqa,UNITED STATES OF AMERICA,25-44,NaN,1.0,1.0,Leisure and Holidays,Conference Tourism,"Radio, TV, Web",Independent,...,No,No,No,No,No,No,11,0,Yes,High Cost
13499,tour_idivz7lpc8,ITALY,25-44,NaN,1.0,1.0,Leisure and Holidays,Beach Tourism,"Travel agent, tour operator",Package Tour,...,Yes,Yes,Yes,No,No,No,0,7,Yes,Normal Cost
15213,tour_idb9xsmbi1,UNITED KINGDOM,25-44,NaN,1.0,1.0,Leisure and Holidays,Wildlife Tourism,"Travel agent, tour operator",Package Tour,...,Yes,Yes,No,Yes,Yes,No,7,0,Yes,Normal Cost
15994,tour_idp90whjre,ITALY,25-44,NaN,1.0,1.0,Leisure and Holidays,Beach Tourism,"Friends, relatives",Package Tour,...,Yes,Yes,Yes,Yes,Yes,Yes,0,7,Yes,Higher Cost
16030,tour_ide70fvj1y,ITALY,45-64,NaN,1.0,1.0,Leisure and Holidays,Beach Tourism,"Travel agent, tour operator",Package Tour,...,Yes,Yes,Yes,Yes,Yes,Yes,11,6,No,Highest Cost
18070,tour_idc45a73cc,SWEDEN,18-24,NaN,1.0,1.0,Leisure and Holidays,Wildlife Tourism,"Travel agent, tour operator",Package Tour,...,No,No,No,No,No,No,32,7,Yes,Normal Cost


test dataframe :


,Tour_ID,country,age_group,travel_with,total_female,total_male,purpose,main_activity,info_source,tour_arrangement,package_transport_int,package_accomodation,package_food,package_transport_tz,package_sightseeing,package_guided_tour,package_insurance,night_mainland,night_zanzibar,first_trip_tz
953,tour_idd0r31y09,EGYPT,25-44,NaN,1.0,1.0,Leisure and Holidays,Wildlife Tourism,"Newspaper, magazines, brochures",Independent,No,No,No,No,No,No,No,23,3,Yes
2096,tour_idobv4yb8i,ITALY,45-64,NaN,1.0,1.0,Leisure and Holidays,Beach Tourism,"Travel agent, tour operator",Package Tour,Yes,Yes,Yes,No,No,No,No,0,7,Yes
2721,tour_idqi6nvow6,CANADA,45-64,NaN,1.0,1.0,Leisure and Holidays,Wildlife Tourism,"Travel agent, tour operator",Package Tour,No,Yes,Yes,Yes,Yes,Yes,No,5,0,No
2776,tour_id0ue6iywk,ITALY,25-44,NaN,1.0,1.0,Leisure and Holidays,Cultural Tourism,"Radio, TV, Web",Package Tour,Yes,Yes,Yes,Yes,Yes,Yes,Yes,7,7,Yes


In [20]:
mask_nan = train_df['travel_with'] == 'With Spouse'
mask_female = train_df['total_female'] == 1 
mask_male   = train_df['total_male'] == 1 

travel_with_spouse = train_df[mask_nan & mask_female & mask_male]
travel_with_spouse.purpose.value_counts()

purpose
Leisure and Holidays              4213
Visiting Friends and Relatives     213
Business                            75
Meetings and Conference             35
Other                               19
Volunteering                        19
Scientific and Academic              5
Medical                              2
Name: count, dtype: int64

so main purpose when travel with spouse is `Leisure and Holidays`

In [21]:
omale_and_ofemale 

NameError: name 'omale_and_ofemale' is not defined

In [ ]:
# with Spouse handel data

def wshd(df):
    mask_nan = df['travel_with'].isna()
    mask_female = df['total_female'] == 1  
    mask_male   = df['total_male'] == 1 
    mask_purpose = df['purpose'].isin(['Leisure and Holidays', 'Visiting Friends and Relatives'])

    mask = mask_nan & mask_female & mask_male & mask_purpose
    df.loc[mask, 'travel_with'] = 'With Spouse' 

wshd(train_df)
wshd(test_df)


In [ ]:
fmva(train_df, test_df)


Missing Values in train dataset : 


,number of missing values,percentage of missing values
Tour_ID,,
country,,
age_group,,
travel_with,21.0,0.11
total_female,2.0,0.01
total_male,6.0,0.03
purpose,,
main_activity,,
info_source,,
tour_arrangement,,


__________________________________________________ 

Missing Values in test dataset : 


,number of missing values,percentage of missing values
Tour_ID,,
country,,
age_group,,
travel_with,3.0,0.05
total_female,2.0,0.03
total_male,1.0,0.02
purpose,,
main_activity,,
info_source,,
tour_arrangement,,


In [ ]:
train_df = train_df.dropna()


In [ ]:
train_df.info()

<class 'pandas.core.frame.DataFrame'>
Index: 18478 entries, 0 to 18505
Data columns (total 21 columns):
 #   Column                 Non-Null Count  Dtype  
---  ------                 --------------  -----  
 0   Tour_ID                18478 non-null  object 
 1   country                18478 non-null  object 
 2   age_group              18478 non-null  object 
 3   travel_with            18478 non-null  object 
 4   total_female           18478 non-null  float64
 5   total_male             18478 non-null  float64
 6   purpose                18478 non-null  object 
 7   main_activity          18478 non-null  object 
 8   info_source            18478 non-null  object 
 9   tour_arrangement       18478 non-null  object 
 10  package_transport_int  18478 non-null  object 
 11  package_accomodation   18478 non-null  object 
 12  package_food           18478 non-null  object 
 13  package_transport_tz   18478 non-null  object 
 14  package_sightseeing    18478 non-null  object 
 15  package

In [ ]:
for col in NUMERIC_COLS:
    if col not in df.columns:
        print('error')
    else:
        print("true")


In [ ]:
fmva(train_df)

Missing Values in train dataset : 


,number of missing values,percentage of missing values
Tour_ID,,
country,,
age_group,,
travel_with,,
total_female,,
total_male,,
purpose,,
main_activity,,
info_source,,
tour_arrangement,,


## 5. Handle Duplicate Records

In [ ]:
print('Number of duplicate records in train dataset:', train_df.duplicated().sum())
print('Number of duplicate records in test dataset:', test_df.duplicated().sum())

Number of duplicate records in train dataset: 0
Number of duplicate records in test dataset: 0


## 6. Standardize Data Types

In [ ]:
train_df.dtypes


Tour_ID                   object
country                   object
age_group                 object
travel_with               object
total_female             float64
total_male               float64
purpose                   object
main_activity             object
info_source               object
tour_arrangement          object
package_transport_int     object
package_accomodation      object
package_food              object
package_transport_tz      object
package_sightseeing       object
package_guided_tour       object
package_insurance         object
night_mainland             int64
night_zanzibar             int64
first_trip_tz             object
cost_category             object
dtype: object

In [ ]:
test_df.dtypes

Tour_ID                   object
country                   object
age_group                 object
travel_with               object
total_female             float64
total_male               float64
purpose                   object
main_activity             object
info_source               object
tour_arrangement          object
package_transport_int     object
package_accomodation      object
package_food              object
package_transport_tz      object
package_sightseeing       object
package_guided_tour       object
package_insurance         object
night_mainland             int64
night_zanzibar             int64
first_trip_tz             object
dtype: object

In [ ]:

data_values_com = pd.DataFrame({
    'train types' : train_df.dtypes,
    'test types' : test_df.dtypes})


In [ ]:

data_values_com

,train types,test types
Tour_ID,object,object
age_group,object,object
cost_category,object,NaN
country,object,object
first_trip_tz,object,object
info_source,object,object
main_activity,object,object
night_mainland,int64,int64
night_zanzibar,int64,int64
package_accomodation,object,object


### check NUMIRCAL COLUMNS (no any nigtive number)

In [ ]:
display(train_df[NUMERIC_COLS].describe().T)
display(test_df[NUMERIC_COLS].describe().T)

,count,mean,std,min,25%,50%,75%,max
total_female,18478.0,0.932839,1.204878,0.0,0.0,1.0,1.0,49.0
total_male,18478.0,0.997727,1.172420,0.0,1.0,1.0,1.0,58.0
night_mainland,18478.0,9.140491,14.132483,0.0,3.0,6.0,11.0,365.0
night_zanzibar,18478.0,2.495671,5.278308,0.0,0.0,0.0,4.0,240.0


,count,mean,std,min,25%,50%,75%,max
total_female,6167.0,0.922491,1.173067,0.0,0.0,1.0,1.0,30.0
total_male,6168.0,1.017510,1.526274,0.0,1.0,1.0,1.0,90.0
night_mainland,6169.0,9.311720,16.366900,0.0,3.0,6.0,11.0,664.0
night_zanzibar,6169.0,2.585832,5.465058,0.0,0.0,0.0,4.0,174.0


In [34]:
for col in train_df.columns:
    print(train_df[col].name,'\n', train_df[col].unique())
    print(train_df[col].dtype)
    print(train_df[col].map(type).value_counts(),'\n')


Tour_ID 
 ['tour_id1hffseyw' 'tour_idnacd7zag' 'tour_id62vz7e71' ...
 'tour_idv7pz3vs8' 'tour_idy6ydo00w' 'tour_idceoq9por']
object
Tour_ID
<class 'str'>    18506
Name: count, dtype: int64 

country 
 ['ITALY' 'UNITED KINGDOM' 'UNITED STATES OF AMERICA' 'RWANDA' 'FRANCE'
 'SWIZERLAND' 'NIGERIA' 'AUSTRALIA' 'SPAIN' 'DENMARK' 'BELGIUM' 'ZAMBIA'
 'NETHERLANDS' 'CANADA' 'INDIA' 'NORWAY' 'ZIMBABWE' 'MORROCO' 'CHINA'
 'GERMANY' 'SOUTH AFRICA' 'JAPAN' 'UAE' 'RUSSIA' 'KENYA' 'GREECE' 'POLAND'
 'AUSTRIA' 'MALAYSIA' 'CZECH REPUBLIC' 'KOREA' 'DRC' 'CYPRUS' 'KUWAIT'
 'BRAZIL' 'CHILE' 'BULGARIA' 'UNITED ARAB EMIRATES' 'ISRAEL' 'NEW ZEALAND'
 'UGANDA' 'IRELAND' 'SWEDEN' 'QATAR' 'CONGO' 'BURUNDI' 'FINLAND' 'GHANA'
 'OMAN' 'LEBANON' 'LUXEMBOURG' 'IVORY COAST' 'NAMIBIA' 'ROMANIA'
 'THAILAND' 'SUDAN' 'SINGAPORE' 'PORTUGAL' 'MALAWI' 'MALT' 'EGYPT'
 'PAKISTAN' 'TANZANIA' 'ARGENTINA' 'SRI LANKA' 'BOTSWANA' 'VIETNAM'
 'BERMUDA' 'TAIWAN' 'SENEGAL' 'AFGHANISTAN' 'COLOMBIA' 'JAMAICA' 'BAHRAIN'
 'COSTARICA' 'CO

`travel_with` in train dataset was have 2 data types, 18478 for string values and 28 for float, mabye the 28 values is missing values so we need to check.

In [36]:
print(train_df["travel_with"].isna().sum())


28


In [44]:

mask = ~train_df['travel_with'].map(lambda value: 
            isinstance(value, str) or pd.isna(value))
train_df.loc[mask, "travel_with"]

Series([], Name: travel_with, dtype: object)

the 28 float values in `travel_with` is actuly missing values.

In [33]:
for col in test_df.columns:
    print(test_df[col].name,'\n', test_df[col].unique())
    print(test_df[col].dtype)
    print(test_df[col].map(type).value_counts(),'\n')


Tour_ID 
 ['tour_idynufedne' 'tour_id9r3y5moe' 'tour_idf6itml6g' ...
 'tour_id7wwqrs0p' 'tour_idx80vbw5a' 'tour_id8fkkwytb']
object
Tour_ID
<class 'str'>    6169
Name: count, dtype: int64 

country 
 ['KOREA' 'UNITED KINGDOM' 'ITALY' 'KENYA' 'ZAMBIA' 'UGANDA' 'NETHERLANDS'
 'OMAN' 'ZIMBABWE' 'BELGIUM' 'DRC' 'UNITED STATES OF AMERICA' 'FRANCE'
 'CANADA' 'CZECH REPUBLIC' 'UAE' 'SPAIN' 'SOUTH AFRICA' 'SWEDEN' 'NORWAY'
 'CHINA' 'GERMANY' 'RWANDA' 'INDIA' 'AUSTRIA' 'DJIBOUT' 'IRELAND'
 'SWIZERLAND' 'AUSTRALIA' 'PHILIPINES' 'BURUNDI' 'JAPAN' 'THAILAND'
 'PORTUGAL' 'DENMARK' 'MOZAMBIQUE' 'UNITED ARAB EMIRATES' 'TAIWAN'
 'BRAZIL' 'HUNGARY' 'SRI LANKA' 'TUNISIA' 'INDONESIA' 'ISRAEL'
 'NEW ZEALAND' 'POLAND' 'CONGO' 'ESTONIA' 'COLOMBIA' 'MALAYSIA' 'TANZANIA'
 'BULGARIA' 'RUSSIA' 'COMORO' 'UKRAIN' 'LEBANON' 'URUGUAY' 'SOMALI'
 'FINLAND' 'ETHIOPIA' 'SCOTLAND' 'TURKEY' 'SLOVENIA' 'MORROCO' 'SWAZILAND'
 'LUXEMBOURG' 'MALAWI' 'SINGAPORE' 'SUDAN' 'PAPUA NEW GUINEA' 'PAKISTAN'
 'EGYPT' 'CYPRUS' 'QATAR' 

also we have 7 float values in `travel_with` but in testing dataset, 
so start check this value again.

In [45]:
print(test_df["travel_with"].isna().sum())


7


In [47]:
mask = ~test_df['travel_with'].map(lambda value: 
            isinstance(value, str) or pd.isna(value))
test_df.loc[mask, "travel_with"]

Series([], Name: travel_with, dtype: object)

also the 7 float values in `travel_with` is missing values 

In [29]:
print('ID data type in train dataset :',train_df[ID_COL].dtype)
print(train_df[ID_COL].map(type).value_counts())
print('_'*25)
print('ID data type in test dataset :',test_df[ID_COL].dtype)
print(test_df[ID_COL].map(type).value_counts())

ID data type in train dataset : object
Tour_ID
<class 'str'>    18506
Name: count, dtype: int64
_________________________
ID data type in test dataset : object
Tour_ID
<class 'str'>    6169
Name: count, dtype: int64


In [ ]:
print("TARGET in train:", "cost_category" in train_df.columns)
print("TARGET in test:", "cost_category" in test_df.columns)

TARGET in train: True
TARGET in test: False


In [24]:
print('TARGET data type in train dataset :',train_df[TARGET_COL].dtype)
print(train_df[TARGET_COL].map(type).value_counts())

TARGET data type in train dataset : object
cost_category
<class 'str'>    18506
Name: count, dtype: int64


In [23]:
display(TARGET_CLASSES)
train_df[TARGET_COL].value_counts()

['Lower Cost',
 'Low Cost',
 'Normal Cost',
 'High Cost',
 'Higher Cost',
 'Highest Cost']

cost_category
Normal Cost     5471
Higher Cost     4865
High Cost       3678
Lower Cost      2567
Low Cost        1566
Highest Cost     359
Name: count, dtype: int64

## 7. Handle Invalid and Inconsistent Values

In [ ]:
display(train_df[NUMERIC_COLS].describe().T)
display(test_df[NUMERIC_COLS].describe().T)

,count,mean,std,min,25%,50%,75%,max
total_female,18478.0,0.932839,1.204878,0.0,0.0,1.0,1.0,49.0
total_male,18478.0,0.997727,1.172420,0.0,1.0,1.0,1.0,58.0
night_mainland,18478.0,9.140491,14.132483,0.0,3.0,6.0,11.0,365.0
night_zanzibar,18478.0,2.495671,5.278308,0.0,0.0,0.0,4.0,240.0


,count,mean,std,min,25%,50%,75%,max
total_female,6167.0,0.922491,1.173067,0.0,0.0,1.0,1.0,30.0
total_male,6168.0,1.017510,1.526274,0.0,1.0,1.0,1.0,90.0
night_mainland,6169.0,9.311720,16.366900,0.0,3.0,6.0,11.0,664.0
night_zanzibar,6169.0,2.585832,5.465058,0.0,0.0,0.0,4.0,174.0


In [ ]:
for col in train_df[CATEGORICAL_COLS]:
    print(train_df[col].name,'\n', train_df[col].unique())
    print(train_df[col].dtype, '\n')

country 
 ['ITALY' 'UNITED KINGDOM' 'UNITED STATES OF AMERICA' 'RWANDA' 'FRANCE'
 'SWIZERLAND' 'NIGERIA' 'AUSTRALIA' 'SPAIN' 'DENMARK' 'BELGIUM' 'ZAMBIA'
 'NETHERLANDS' 'CANADA' 'INDIA' 'NORWAY' 'ZIMBABWE' 'MORROCO' 'CHINA'
 'GERMANY' 'SOUTH AFRICA' 'JAPAN' 'UAE' 'RUSSIA' 'KENYA' 'GREECE' 'POLAND'
 'AUSTRIA' 'MALAYSIA' 'CZECH REPUBLIC' 'KOREA' 'DRC' 'CYPRUS' 'KUWAIT'
 'BRAZIL' 'CHILE' 'BULGARIA' 'UNITED ARAB EMIRATES' 'ISRAEL' 'NEW ZEALAND'
 'UGANDA' 'IRELAND' 'SWEDEN' 'QATAR' 'CONGO' 'BURUNDI' 'FINLAND' 'GHANA'
 'OMAN' 'LEBANON' 'LUXEMBOURG' 'IVORY COAST' 'NAMIBIA' 'ROMANIA'
 'THAILAND' 'SUDAN' 'SINGAPORE' 'PORTUGAL' 'MALAWI' 'MALT' 'EGYPT'
 'PAKISTAN' 'TANZANIA' 'ARGENTINA' 'SRI LANKA' 'BOTSWANA' 'VIETNAM'
 'BERMUDA' 'TAIWAN' 'SENEGAL' 'AFGHANISTAN' 'COLOMBIA' 'JAMAICA' 'BAHRAIN'
 'COSTARICA' 'COMORO' 'URUGUAY' 'SAUD ARABIA' 'ETHIOPIA' 'PHILIPINES'
 'ARMENIA' 'SCOTLAND' 'MEXICO' 'MADAGASCAR' 'SERBIA' 'SLOVAKIA' 'IRAN'
 'SWAZILAND' 'PERU' 'LITHUANIA' 'ALGERIA' 'SEYCHELLES' 'MOZAMBIQUE

In [ ]:
for col in test_df[CATEGORICAL_COLS]:
    print(test_df[col].name,'\n', test_df[col].unique())
    print(test_df[col].dtype, '\n')

country 
 ['KOREA' 'UNITED KINGDOM' 'ITALY' 'KENYA' 'ZAMBIA' 'UGANDA' 'NETHERLANDS'
 'OMAN' 'ZIMBABWE' 'BELGIUM' 'DRC' 'UNITED STATES OF AMERICA' 'FRANCE'
 'CANADA' 'CZECH REPUBLIC' 'UAE' 'SPAIN' 'SOUTH AFRICA' 'SWEDEN' 'NORWAY'
 'CHINA' 'GERMANY' 'RWANDA' 'INDIA' 'AUSTRIA' 'DJIBOUT' 'IRELAND'
 'SWIZERLAND' 'AUSTRALIA' 'PHILIPINES' 'BURUNDI' 'JAPAN' 'THAILAND'
 'PORTUGAL' 'DENMARK' 'MOZAMBIQUE' 'UNITED ARAB EMIRATES' 'TAIWAN'
 'BRAZIL' 'HUNGARY' 'SRI LANKA' 'TUNISIA' 'INDONESIA' 'ISRAEL'
 'NEW ZEALAND' 'POLAND' 'CONGO' 'ESTONIA' 'COLOMBIA' 'MALAYSIA' 'TANZANIA'
 'BULGARIA' 'RUSSIA' 'COMORO' 'UKRAIN' 'LEBANON' 'URUGUAY' 'SOMALI'
 'FINLAND' 'ETHIOPIA' 'SCOTLAND' 'TURKEY' 'SLOVENIA' 'MORROCO' 'SWAZILAND'
 'LUXEMBOURG' 'MALAWI' 'SINGAPORE' 'SUDAN' 'PAPUA NEW GUINEA' 'PAKISTAN'
 'EGYPT' 'CYPRUS' 'QATAR' 'CAMEROON' 'ANGOLA' 'SAUD ARABIA' 'GREECE'
 'KAZAKHSTAN' 'MADAGASCAR' 'CAMBODIA' 'BENIN' 'ARGENTINA' 'VIETNAM' 'PERU'
 'COSTARICA' 'ROMANIA' 'MEXICO' 'DOMINICA' 'GHANA' 'BOSNIA' 'ECUADO'
 'B

1- The country column contains duplicate representations (like 'UAE' and 'UNITED ARAB EMIRATES'), one country that is part of another ('SCOTLAND' is part of the 'UNITED KINGDOM'), and 12 countries with spelling mistakes in train dataset and 11 in test dataset.

In [ ]:
train_df['country'] = train_df['country'].replace({
    'UNITED ARAB EMIRATES':'UAE', # is same country
    'BURGARIA' : 'BULGARIA', # is same country
    
    'SCOTLAND': 'UNITED KINGDOM', # scotland it's in united kingdom 

    # Spelling mistakes    
    'SWIZERLAND': 'SWITZERLAND',
    'DJIBOUT': 'DJIBOUTI',
    'PHILIPINES': 'PHILIPPINES',
    'UKRAIN': 'UKRAINE',
    'SOMALI': 'SOMALIA',
    'MORROCO': 'MOROCCO',
    'SAUD ARABIA': 'SAUDI ARABIA',
    'COSTARICA': 'COSTA RICA',
    'COMORO': 'COMOROS',
    'ECUADO': 'ECUADOR',
    'MALT': 'MALTA',
    'TRINIDAD TOBACCO': 'TRINIDAD AND TOBAGO' # only in train data

})
test_df['country'] = test_df['country'].replace({
    'UNITED ARAB EMIRATES':'UAE', # is same country
    'BURGARIA' : 'BULGARIA', # is same country
    
    'SCOTLAND': 'UNITED KINGDOM', # scotland it's in united kingdom 

    # Spelling mistakes
    'SWIZERLAND': 'SWITZERLAND',
    'DJIBOUT': 'DJIBOUTI',
    'PHILIPINES': 'PHILIPPINES',
    'UKRAIN': 'UKRAINE',
    'SOMALI': 'SOMALIA',
    'MORROCO': 'MOROCCO',
    'SAUD ARABIA': 'SAUDI ARABIA',
    'COSTARICA': 'COSTA RICA',
    'COMORO': 'COMOROS',
    'ECUADO': 'ECUADOR',
    'MALT': 'MALTA'
})

### 2- Widlife Tourism and Wildlife Tourism 

In [ ]:
train_df['main_activity'].value_counts()

main_activity
Widlife Tourism             5988
Beach Tourism               4170
Wildlife Tourism            2264
Conference Tourism          1917
Hunting Tourism             1710
Cultural Tourism            1443
Mountain Climbing            528
Business                     280
Bird Tourism                 144
Diving and Sport Fishing      34
Name: count, dtype: int64

duplicated value in `main_activity` column, `Widlife Tourism` with 5988 and `Wildlife Tourism` with 2264 

In [ ]:
train_df['main_activity'] = train_df['main_activity'].replace({'Widlife Tourism':'Wildlife Tourism'})
test_df['main_activity'] = test_df['main_activity'].replace({'Widlife Tourism':'Wildlife Tourism'})

In [ ]:
train_df['main_activity'].value_counts()

main_activity
Wildlife Tourism            8252
Beach Tourism               4170
Conference Tourism          1917
Hunting Tourism             1710
Cultural Tourism            1443
Mountain Climbing            528
Business                     280
Bird Tourism                 144
Diving and Sport Fishing      34
Name: count, dtype: int64

In [ ]:
test_df['main_activity'].value_counts()

main_activity
Wildlife Tourism            2779
Beach Tourism               1432
Conference Tourism           612
Hunting Tourism              523
Cultural Tourism             495
Mountain Climbing            198
Business                      90
Bird Tourism                  34
Diving and Sport Fishing       6
Name: count, dtype: int64

## 8. Investigate and Handle Outliers

In [ ]:
train_df[NUMERIC_COLS].describe(percentiles=[0.90, 0.95, 0.99, 0.995]).T

,count,mean,std,min,50%,90%,95%,99%,99.5%,max
total_female,18478.0,0.932839,1.204878,0.0,1.0,2.0,3.0,5.0,6.0,49.0
total_male,18478.0,0.997727,1.172420,0.0,1.0,2.0,2.0,5.0,6.0,58.0
night_mainland,18478.0,9.140491,14.132483,0.0,6.0,17.0,28.0,61.0,90.0,365.0
night_zanzibar,18478.0,2.495671,5.278308,0.0,0.0,7.0,10.0,17.0,22.0,240.0


## 9. Validate Preprocessing Decisions

In [ ]:
print(f'shape of train dataset : {train_df.shape}')
print(f'rows in train dataset : {len(train_df)}')
print(f'columns in train dataset : {train_df.shape[1]}')
print("_"*50,'\n')
print(f'shape of test dataset : {test_df.shape}')
print(f'rows in test dataset : {len(test_df)}')
print(f'columns in test dataset : {test_df.shape[1]}')


shape of train dataset : (18478, 21)
rows in train dataset : 18478
columns in train dataset : 21
__________________________________________________ 

shape of test dataset : (6169, 20)
rows in test dataset : 6169
columns in test dataset : 20


In [ ]:
fmva(train_df, test_df)

Missing Values in train dataset : 


,number of missing values,percentage of missing values
Tour_ID,,
country,,
age_group,,
travel_with,,
total_female,,
total_male,,
purpose,,
main_activity,,
info_source,,
tour_arrangement,,


__________________________________________________ 

Missing Values in test dataset : 


,number of missing values,percentage of missing values
Tour_ID,,
country,,
age_group,,
travel_with,3.0,0.05
total_female,2.0,0.03
total_male,1.0,0.02
purpose,,
main_activity,,
info_source,,
tour_arrangement,,


- test dataset will not saved, and the method we have apply to missing values in test data its only for understand how we will do with missing data in test stage   

In [ ]:
print(f'duplicated row in train dataset is {train_df.duplicated().sum()}')
print(f'duplicated row in test dataset is {test_df.duplicated().sum()}')


duplicated row in train dataset is 0
duplicated row in test dataset is 0


In [ ]:
print(f'duplicated ID column in train dataset is {train_df[ID_COL].duplicated().sum()}')
print(f'duplicated ID column in test dataset is {test_df[ID_COL].duplicated().sum()}')


duplicated ID column in train dataset is 0
duplicated ID column in test dataset is 0


In [ ]:
data_values_com = pd.DataFrame({
    'train types' : train_df.dtypes,
    'test types' : test_df.dtypes})
data_values_com

,train types,test types
Tour_ID,object,object
age_group,object,object
cost_category,object,NaN
country,object,object
first_trip_tz,object,object
info_source,object,object
main_activity,object,object
night_mainland,int64,int64
night_zanzibar,int64,int64
package_accomodation,object,object


In [ ]:
for col in train_df[CATEGORICAL_COLS]:
    print(train_df[col].name,'\n', train_df[col].unique())
    print(train_df[col].dtype, '\n')

country 
 ['ITALY' 'UNITED KINGDOM' 'UNITED STATES OF AMERICA' 'RWANDA' 'FRANCE'
 'SWIZERLAND' 'NIGERIA' 'AUSTRALIA' 'SPAIN' 'DENMARK' 'BELGIUM' 'ZAMBIA'
 'NETHERLANDS' 'CANADA' 'INDIA' 'NORWAY' 'ZIMBABWE' 'MORROCO' 'CHINA'
 'GERMANY' 'SOUTH AFRICA' 'JAPAN' 'UAE' 'RUSSIA' 'KENYA' 'GREECE' 'POLAND'
 'AUSTRIA' 'MALAYSIA' 'CZECH REPUBLIC' 'KOREA' 'DRC' 'CYPRUS' 'KUWAIT'
 'BRAZIL' 'CHILE' 'BURGARIA' 'ISRAEL' 'NEW ZEALAND' 'UGANDA' 'IRELAND'
 'SWEDEN' 'QATAR' 'CONGO' 'BURUNDI' 'FINLAND' 'GHANA' 'OMAN' 'LEBANON'
 'LUXEMBOURG' 'IVORY COAST' 'NAMIBIA' 'ROMANIA' 'THAILAND' 'SUDAN'
 'SINGAPORE' 'PORTUGAL' 'MALAWI' 'MALT' 'EGYPT' 'PAKISTAN' 'TANZANIA'
 'ARGENTINA' 'SRI LANKA' 'BOTSWANA' 'VIETNAM' 'BERMUDA' 'TAIWAN' 'SENEGAL'
 'AFGHANISTAN' 'COLOMBIA' 'JAMAICA' 'BAHRAIN' 'COSTARICA' 'COMORO'
 'URUGUAY' 'SAUD ARABIA' 'ETHIOPIA' 'PHILIPINES' 'ARMENIA' 'SCOTLAND'
 'MEXICO' 'MADAGASCAR' 'SERBIA' 'SLOVAKIA' 'IRAN' 'SWAZILAND' 'PERU'
 'LITHUANIA' 'ALGERIA' 'SEYCHELLES' 'MOZAMBIQUE' 'UKRAIN' 'NEPAL'
 'CR

In [ ]:
for col in test_df[CATEGORICAL_COLS]:
    print(test_df[col].name,'\n', test_df[col].unique())
    print(test_df[col].dtype, '\n')

country 
 ['KOREA' 'UNITED KINGDOM' 'ITALY' 'KENYA' 'ZAMBIA' 'UGANDA' 'NETHERLANDS'
 'OMAN' 'ZIMBABWE' 'BELGIUM' 'DRC' 'UNITED STATES OF AMERICA' 'FRANCE'
 'CANADA' 'CZECH REPUBLIC' 'UNITED ARAB EMIRATES' 'SPAIN' 'SOUTH AFRICA'
 'SWEDEN' 'NORWAY' 'CHINA' 'GERMANY' 'RWANDA' 'INDIA' 'AUSTRIA' 'DJIBOUT'
 'IRELAND' 'SWIZERLAND' 'AUSTRALIA' 'PHILIPINES' 'BURUNDI' 'JAPAN'
 'THAILAND' 'PORTUGAL' 'DENMARK' 'MOZAMBIQUE' 'TAIWAN' 'BRAZIL' 'HUNGARY'
 'SRI LANKA' 'TUNISIA' 'INDONESIA' 'ISRAEL' 'NEW ZEALAND' 'POLAND' 'CONGO'
 'ESTONIA' 'COLOMBIA' 'MALAYSIA' 'TANZANIA' 'BULGARIA' 'RUSSIA' 'COMORO'
 'UKRAIN' 'LEBANON' 'URUGUAY' 'SOMALI' 'FINLAND' 'ETHIOPIA' 'SCOTLAND'
 'TURKEY' 'SLOVENIA' 'MORROCO' 'SWAZILAND' 'LUXEMBOURG' 'MALAWI'
 'SINGAPORE' 'SUDAN' 'PAPUA NEW GUINEA' 'PAKISTAN' 'EGYPT' 'CYPRUS'
 'QATAR' 'CAMEROON' 'ANGOLA' 'SAUD ARABIA' 'GREECE' 'KAZAKHSTAN'
 'MADAGASCAR' 'CAMBODIA' 'BENIN' 'ARGENTINA' 'VIETNAM' 'PERU' 'COSTARICA'
 'ROMANIA' 'MEXICO' 'DOMINICA' 'GHANA' 'BOSNIA' 'ECUADO' 'BERMUDA'

In [ ]:
def inf(df):
    infinite_values = np.isinf(df[NUMERIC_COLS]).sum()
    print(f'Infinite values : \n{infinite_values}')
    display(df[NUMERIC_COLS].describe().T)


In [ ]:
inf(train_df)

Infinite values : 
total_female      0
total_male        0
night_mainland    0
night_zanzibar    0
dtype: int64


,count,mean,std,min,25%,50%,75%,max
total_female,18478.0,0.932839,1.204878,0.0,0.0,1.0,1.0,49.0
total_male,18478.0,0.997727,1.172420,0.0,1.0,1.0,1.0,58.0
night_mainland,18478.0,9.140491,14.132483,0.0,3.0,6.0,11.0,365.0
night_zanzibar,18478.0,2.495671,5.278308,0.0,0.0,0.0,4.0,240.0


In [ ]:
inf(test_df)

Infinite values : 
total_female      0
total_male        0
night_mainland    0
night_zanzibar    0
dtype: int64


,count,mean,std,min,25%,50%,75%,max
total_female,6167.0,0.922491,1.173067,0.0,0.0,1.0,1.0,30.0
total_male,6168.0,1.017510,1.526274,0.0,1.0,1.0,1.0,90.0
night_mainland,6169.0,9.311720,16.366900,0.0,3.0,6.0,11.0,664.0
night_zanzibar,6169.0,2.585832,5.465058,0.0,0.0,0.0,4.0,174.0


In [ ]:
print(f'missing values in TARGET column : {train_df[TARGET_COL].isnull().sum()}')
print(f'\nTARGET class : \n{train_df[TARGET_COL].value_counts()}')


missing values in TARGET column : 0

TARGET class : 
cost_category
Normal Cost     5465
Higher Cost     4858
High Cost       3672
Lower Cost      2563
Low Cost        1564
Highest Cost     356
Name: count, dtype: int64


## 10. Preprocessing Summary